# AI Programming — Lecture 19
## Lab 4-1: Transformer Encoder for Text Classification

17–18강에서 배운 **Transformer Encoder**를 실제 텍스트 분류 문제에 적용합니다.

같은 encoder 구조를 두 개의 dataset에 사용합니다.

- **IMDB**: 영화 리뷰의 긍정/부정 이진 분류
- **Reuters**: 뉴스 기사의 46-class 분류

### 학습 목표
- Token embedding과 positional embedding을 함께 사용할 수 있습니다.
- Multi-Head Self-Attention과 FFN으로 Transformer Encoder block을 구성할 수 있습니다.
- Padding mask를 attention과 pooling에 반영할 수 있습니다.
- 같은 Transformer Encoder를 binary / multiclass classification에 재사용할 수 있습니다.

### 주요 설정
```text
Sequence length: 200
Embedding dim  : 32
Attention heads: 2
FFN dim        : 64
```

> 이론은 17–18강에서 다루었으므로, 이번 실습에서는 **코드에서 각 구성 요소가 어디에 해당하는지** 확인하는 데 집중합니다.

## 1. 실습 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb, reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences

tf.keras.utils.set_random_seed(42)

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64

## 2. Transformer Encoder 구성

먼저 세 가지 핵심 layer를 정의합니다.

1. **Token + Position Embedding**
2. **Transformer Encoder Block**
3. **Masked Global Max Pooling**

Padding token은 실제 단어가 아니므로 attention과 pooling에서 제외합니다.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.supports_masking = True

        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(max_len, embed_dim)

    def call(self, inputs):
        positions = keras.ops.arange(0, keras.ops.shape(inputs)[1], 1)
        return self.token_embedding(inputs) + self.position_embedding(positions)

    def compute_mask(self, inputs, mask=None):
        return keras.ops.not_equal(inputs, 0)


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.supports_masking = True

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, mask=None):
        attention_mask = None
        if mask is not None:
            attention_mask = keras.ops.expand_dims(mask, axis=1)

        attention_output = self.attention(inputs, inputs, attention_mask=attention_mask)
        x = self.norm1(inputs + self.dropout1(attention_output))

        ffn_output = self.ffn(x)
        return self.norm2(x + self.dropout2(ffn_output))

    def compute_mask(self, inputs, mask=None):
        return mask


class MaskedGlobalMaxPooling1D(layers.Layer):
    def call(self, inputs, mask=None):
        if mask is not None:
            mask = keras.ops.expand_dims(mask, axis=-1)
            inputs = keras.ops.where(mask, inputs, keras.ops.cast(-1e9, inputs.dtype))

        return keras.ops.max(inputs, axis=1)

### Classification Model

전체 흐름은 다음과 같습니다.

```text
Token IDs
→ Token + Position Embedding
→ Transformer Encoder
→ Masked Global Max Pooling
→ Dropout
→ Classifier
```

IMDB에서는 `sigmoid`, Reuters에서는 `softmax` output을 사용합니다.

In [ ]:
def build_classifier(num_classes):
    inputs = keras.Input(shape=(MAX_LEN,), dtype="int32")

    embedding_layer = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)
    x = embedding_layer(inputs)

    encoder_layer = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM)
    x = encoder_layer(x)

    pooling_layer = MaskedGlobalMaxPooling1D()
    x = pooling_layer(x)

    dropout_layer = layers.Dropout(0.2)
    x = dropout_layer(x)

    if num_classes == 2:
        output_layer = layers.Dense(1, activation="sigmoid")
    else:
        output_layer = layers.Dense(num_classes, activation="softmax")

    outputs = output_layer(x)

    model = keras.Model(inputs, outputs)
    return model

## 3. IMDB — Binary Classification

IMDB 영화 리뷰를 Positive / Negative로 분류합니다.

- Output: 1
- Activation: Sigmoid
- Loss: Binary Cross Entropy

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

In [ ]:
imdb_model = build_classifier(num_classes=2)

imdb_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

imdb_model.summary()

In [ ]:
imdb_early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

imdb_history = imdb_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[imdb_early_stopping],
    verbose=1
)

In [ ]:
test_loss, test_acc = imdb_model.evaluate(
    X_test, y_test, verbose=0
)

print(f"IMDB Test Accuracy: {test_acc:.4f}")

### Learning Curve

Training loss와 validation loss를 비교하여 수렴과 overfitting 여부를 확인합니다.

In [ ]:
plt.plot(imdb_history.history["loss"], label="Train")
plt.plot(imdb_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 4. Reuters — Multiclass Classification

Reuters 뉴스 기사를 46개의 topic으로 분류합니다.

- Output: 46
- Activation: Softmax
- Loss: Sparse Categorical Cross Entropy

In [ ]:
(X_train_r, y_train_r), (X_test_r, y_test_r) = reuters.load_data(num_words=VOCAB_SIZE)

X_train_r = pad_sequences(
    X_train_r,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_r = pad_sequences(
    X_test_r,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

NUM_CLASSES = int(np.max(y_train_r)) + 1

print("Train:", X_train_r.shape, y_train_r.shape)
print("Test :", X_test_r.shape, y_test_r.shape)
print("Classes:", NUM_CLASSES)

In [ ]:
reuters_model = build_classifier(num_classes=NUM_CLASSES)

reuters_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

reuters_model.summary()

In [ ]:
reuters_early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

reuters_history = reuters_model.fit(
    X_train_r,
    y_train_r,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[reuters_early_stopping],
    verbose=1
)

In [ ]:
test_loss, test_acc = reuters_model.evaluate(
    X_test_r, y_test_r, verbose=0
)

print(f"Reuters Test Accuracy: {test_acc:.4f}")

In [ ]:
plt.plot(reuters_history.history["loss"], label="Train")
plt.plot(reuters_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 5. 정리

같은 Transformer Encoder를 사용하더라도 **마지막 classifier만 task에 맞게 변경**할 수 있습니다.

```text
IMDB
→ Encoder
→ Dense(1, sigmoid)

Reuters
→ Encoder
→ Dense(46, softmax)
```

### 확인할 내용
1. Positional embedding이 없다면 token의 순서 정보는 어떻게 전달될까요?
2. Padding mask는 왜 필요한가요?
3. Encoder의 출력 sequence를 하나의 vector로 만드는 pooling은 어떤 역할을 하나요?
4. Binary와 multiclass classification에서 마지막 layer가 어떻게 달라지나요?